# 05 — Accessibility

Computes transit accessibility and walkability features per census tract from OSM.

**Data source:** Overpass API — fully portable.

**Method:** Three batch Overpass queries (subway stops, bus stops, street nodes in the study area), then BallTree matching to compute per-tract distances and intersection density.

**Output columns:** `tract_id`, `dist_subway_mean`, `dist_bus_mean`, `transit_stop_count`, `intersection_density`

**Output file:** `csv/05_accessibility.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
SEARCH_RADIUS = 800   # meters — max distance to consider for transit stops
INTERSECTION_RADIUS = 500  # meters — radius for intersection density

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

# Bounding box for batch queries (with buffer)
BUFFER = 0.015  # ~1.5 km buffer around study area
LAT_MIN = df_tracts["tract_lat"].min() - BUFFER
LAT_MAX = df_tracts["tract_lat"].max() + BUFFER
LON_MIN = df_tracts["tract_lon"].min() - BUFFER
LON_MAX = df_tracts["tract_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"
print(f"Bounding box: {BBOX}")

In [ ]:
# ── Overpass batch query helper ───────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            import time
            time.sleep(5 + attempt * 3)
    raise RuntimeError(f"Overpass failed: {last_error}")


def extract_points(data):
    """Extract (lat, lon) from Overpass response elements."""
    points = []
    for el in data.get("elements", []):
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        if lat and lon:
            points.append((float(lat), float(lon)))
    return points


print("Helpers ready.")

In [ ]:
# ── Batch query: ALL subway entrances in study area ───

subway_query = (f'[out:json][timeout:90];\n'
                f'(node["railway"="subway_entrance"]({BBOX});\n'
                f' node["station"="subway"]({BBOX}););\n'
                f'out;')

print("Querying all subway stops...")
subway_data = query_overpass_cached(subway_query)
subway_points = extract_points(subway_data)
print(f"  Found {len(subway_points)} subway entrances/stations")

# ── Batch query: ALL bus stops in study area ──────────

bus_query = (f'[out:json][timeout:90];\n'
             f'(node["highway"="bus_stop"]({BBOX});\n'
             f' node["public_transport"="platform"]["bus"="yes"]({BBOX}););\n'
             f'out;')

print("Querying all bus stops...")
bus_data = query_overpass_cached(bus_query)
bus_points = extract_points(bus_data)
print(f"  Found {len(bus_points)} bus stops")

In [ ]:
# ── BallTree matching: tract centroids → transit stops ─

EARTH_RADIUS_M = 6371000
search_rad = SEARCH_RADIUS / EARTH_RADIUS_M  # convert meters to radians

tract_coords_rad = np.radians(df_tracts[["tract_lat", "tract_lon"]].values)

records = []

# Build BallTrees for subway and bus
if subway_points:
    subway_tree = BallTree(np.radians(subway_points), metric="haversine")
else:
    subway_tree = None

if bus_points:
    bus_tree = BallTree(np.radians(bus_points), metric="haversine")
else:
    bus_tree = None

for i, row in df_tracts.iterrows():
    tract_id = row["tract_id"]
    coord = tract_coords_rad[i].reshape(1, -1)
    
    # Subway: distances to all stops within radius
    if subway_tree is not None:
        idx, dist = subway_tree.query_radius(coord, r=search_rad, return_distance=True)
        subway_dists_m = dist[0] * EARTH_RADIUS_M
        dist_subway_mean = round(float(np.mean(subway_dists_m)), 1) if len(subway_dists_m) > 0 else np.nan
        subway_count = len(subway_dists_m)
    else:
        dist_subway_mean = np.nan
        subway_count = 0
    
    # Bus: distances to all stops within radius
    if bus_tree is not None:
        idx, dist = bus_tree.query_radius(coord, r=search_rad, return_distance=True)
        bus_dists_m = dist[0] * EARTH_RADIUS_M
        dist_bus_mean = round(float(np.mean(bus_dists_m)), 1) if len(bus_dists_m) > 0 else np.nan
        bus_count = len(bus_dists_m)
    else:
        dist_bus_mean = np.nan
        bus_count = 0
    
    records.append({
        "tract_id": tract_id,
        "dist_subway_mean": dist_subway_mean,
        "dist_bus_mean": dist_bus_mean,
        "transit_stop_count": subway_count + bus_count,
    })

df_transit = pd.DataFrame(records)
print(f"Transit data computed for {len(df_transit)} tracts")
print(f"  Tracts with subway access: {df_transit['dist_subway_mean'].notna().sum()}")
print(f"  Tracts with bus access: {df_transit['dist_bus_mean'].notna().sum()}")

In [ ]:
# ── Intersection density via Overpass (highway nodes) ──

# Query all highway nodes in the study area — these approximate street intersections
highway_query = (f'[out:json][timeout:120];\n'
                 f'way["highway"~"^(primary|secondary|tertiary|residential|'
                 f'living_street|pedestrian|unclassified|trunk|service)$"]({BBOX});\n'
                 f'node(w)({BBOX});\n'
                 f'out;')

print("Querying street network nodes...")
highway_data = query_overpass_cached(highway_query)
highway_points = extract_points(highway_data)
print(f"  Found {len(highway_points)} street nodes")

# Build BallTree from street nodes
if highway_points:
    node_tree = BallTree(np.radians(highway_points), metric="haversine")

    int_rad = INTERSECTION_RADIUS / EARTH_RADIUS_M
    AREA_KM2 = math.pi * (INTERSECTION_RADIUS / 1000) ** 2

    int_densities = []
    for i in range(len(df_tracts)):
        coord = tract_coords_rad[i].reshape(1, -1)
        count = node_tree.query_radius(coord, r=int_rad, count_only=True)[0]
        int_densities.append(round(count / AREA_KM2, 1))
else:
    int_densities = [0.0] * len(df_tracts)

df_transit["intersection_density"] = int_densities
print(f"Intersection density computed for {len(df_transit)} tracts")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/05_accessibility.csv"
df_transit.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_transit)} rows x {df_transit.shape[1]} cols)")
print(df_transit.describe().round(1).to_string())